# x3_infracoes — Autos de Infração IBAMA (Código Informado, Municipal-Anual)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** autos de infração (IBAMA), quantidade, municipal pelo código informado, anual  
**Conceito e Mecanismo Econômico:** Contagem e valor de autos de infração ambiental lavrados pelo IBAMA, agregados pelo código municipal declarado no registro. Proxy de rigor institucional e fiscalização ambiental.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__autos_multas_ibama__municipio_anual_candidato.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - n_autos_ibama_it já está agregado por município e ano; não anualizar novamente.
> - Validar tipos pertinentes, status e cobertura; ausência de linha não foi convertida em zero.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código IBGE declarado no auto (141 municípios).
- **`ano`**: Ano da autuação.
- **`n_autos_ibama_it`**: Número de autos lavrados no município e ano.
- **`valor_multas_ibama_it`**: Valor nominal das multas (R$).


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__autos_multas_ibama__municipio_anual_candidato.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x3_infracoes`:
- **Atenção:** n_autos_ibama_it já está agregado por município e ano; não anualizar novamente.
- **Atenção:** Validar tipos pertinentes, status e cobertura; ausência de linha não foi convertida em zero.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')
df['n_autos_ibama_it'] = pd.to_numeric(df['n_autos_ibama_it'], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# Aglutinar registros remapeados
df = df.groupby(['codigo_ibge', 'ano'], as_index=False).agg({
    'n_autos_ibama_it': 'sum',
    'valor_multas_ibama_it': 'sum'
})
print(f"Municípios com autuação registrada: {df['codigo_ibge'].nunique()}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva de autos lavrados por município-ano:")
print(df['n_autos_ibama_it'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

# Alerta econométrico: Na base bruta, ausência de linha não foi preenchida com zero!
print(f"Total de registros na tabela bruta: {len(df)}")

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[df['n_autos_ibama_it'] > 0]['n_autos_ibama_it'], bins=30, log_scale=True, ax=axes[0], color='navy')
axes[0].set_title("Distribuição dos Autos de Infração (Escala Log)")

df_tempo = df.groupby('ano')['n_autos_ibama_it'].sum().reset_index()
sns.lineplot(data=df_tempo, x='ano', y='n_autos_ibama_it', marker='o', ax=axes[1], color='darkblue')
axes[1].set_title("Total Anual de Autos Lavrados em MT")
plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
plt.figure(figsize=(10, 4))
sns.lineplot(data=df_painel, x='ano', y='n_autos_ibama_it', estimator='mean', color='darkblue')
plt.title("Média Municipal de Autos de Infração em MT (2001-2024)")
plt.ylabel("Média de Autos")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Fase 6: Feature Engineering — Autuações Ambientais Ibama e Grade Balanceada

### 1. Step 1: Agregação Anual e Diferenciação Crucial entre Ausência e Zero
- **Zero Verdadeiro:** Se o Ibama não lavrou nenhum auto de infração de flora em um determinado município em 2012, isso significa **zero fiscalização real registrada** naquele ano, e NUNCA dado faltante (`NaN`)!
- **Grade Balanceada Completa:** Criamos o produto cartesiano de 141 municípios $\times$ 24 anos (2001–2024) = 3.384 linhas. Onde não há linha na tabela do Ibama, preenchemos com 0.

### 2. Step 2: Derivações e Intensidade de Fiscalização
- **Defasagem $t-1$:** O número de autos aplicados no ano passado ($t-1$) condiciona a percepção de risco e dissuade o desmatamento no ano corrente ($t$).
- **Taxa por $1.000\text{ km}^2$:** Controla o tamanho geográfico do município, medindo a densidade da presença policial em campo.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Criar Grade Completa Balanceada: 141 municípios x 24 anos (2001-2024) = 3.384 linhas
mun_mt = df['codigo_ibge'].unique()
anos_painel = range(2001, 2025)
grade_completa = pd.MultiIndex.from_product([mun_mt, anos_painel], names=['codigo_ibge', 'ano']).to_frame().reset_index(drop=True)

# 2. Fazer o merge com os dados agregados do Ibama
painel_autos = pd.merge(grade_completa, df, on=['codigo_ibge', 'ano'], how='left')

# 3. Preencher Zeros Verdadeiros (onde não houve registro de autuação pelo Ibama)
# n_autos_ibama_it: quantidade anual de autos de infração ambiental lavrados no município
painel_autos['n_autos_ibama_it'] = painel_autos['n_autos_ibama_it'].fillna(0)

painel_autos.sort_values(['codigo_ibge', 'ano'], inplace=True)

# 4. Defasagem Temporal de 1 ano (t-1)
# n_autos_lag1: fiscalização ambiental ocorrida no ano anterior (efeito dissuasório)
painel_autos['n_autos_lag1'] = painel_autos.groupby('codigo_ibge')['n_autos_ibama_it'].shift(1)

# 5. Transformação Logarítmica para lidar com assimetria
painel_autos['ln_autos_lag1'] = np.log1p(painel_autos['n_autos_lag1'])

print(f"Dimensões da grade balanceada de autos: {painel_autos.shape} (Esperado: exatamente 3.384 linhas)")
print(f"Percentual de observações com zero autuações: {(painel_autos['n_autos_ibama_it'] == 0).mean():.2%}")
print(painel_autos[['codigo_ibge', 'ano', 'n_autos_ibama_it', 'n_autos_lag1', 'ln_autos_lag1']].head())